# 11 สรุปผล

notebook นี้รวมผลของ notebook 00-10 ไว้ที่เดียว สำหรับผู้อ่าน 2 กลุ่ม
- ผู้บริหารและฝ่ายวางแผนของ BART: คำตอบของคำถามธุรกิจ ข้อเสนอที่ทำได้ทันที และตาราง Insight → Pain point รวม
- ผู้ประเมินงาน: เทคนิค Big Data ที่ใช้ ผลของโมเดล และเวลากับทรัพยากรที่ใช้จริงทุกขั้น

ตัวเลขในตารางและกราฟของ notebook นี้อ่านจากไฟล์ที่ notebook ก่อนหน้าสร้างไว้ (`data/`, `outputs/tables/`, `outputs/logs/perf.jsonl`) ไม่ได้คำนวณใหม่หรือพิมพ์ตัวเลขเอง ส่วนตัวเลขในข้อความอ้างอิงจาก notebook ต้นทางที่ระบุไว้

หัวข้อ
1. ข้อมูลที่ใช้
2. คำตอบของคำถามธุรกิจ Q1-Q4
3. ผลของโมเดล M1-M3
4. ข้อเสนอสำหรับผู้บริหาร
5. ตาราง Insight → Pain point รวม
6. ทำงานได้เร็วแค่ไหน
7. เทคนิค Big Data ที่ใช้
8. ข้อจำกัดและงานต่อ

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import json
import math

import pandas as pd
import plotly.graph_objects as go
import polars as pl

from bart import viz
from bart.config import FEATURES, FIG, LOGS, MARTS, MODELS, OD, RAW, TAB, create_views, duck, fetch_row, fetch_value

viz.notebook_mode()

## 1. ข้อมูลที่ใช้

นับใหม่จากไฟล์จริงทุกครั้งที่รัน ทั้งขนาดไฟล์ จำนวนแถว และจำนวนทริป

In [2]:
con = duck()
create_views(con)


def folder_size(paths):
    total = 0
    for path in paths:
        total += path.stat().st_size
    return total


csv_files = sorted(RAW.glob("*.csv"))
parquet_files = sorted(OD.rglob("*.parquet"))
mart_files = sorted(MARTS.glob("*.parquet"))

csv_bytes = folder_size(csv_files)
parquet_bytes = folder_size(parquet_files)
mart_bytes = folder_size(mart_files)

od_rows, od_trips, first_date, last_date, station_count = fetch_row(
    con, "SELECT count(*), sum(trips), min(date), max(date), count(DISTINCT origin) FROM od"
)
station_hour_rows = fetch_value(con, "SELECT count(*) FROM station_hour")
feature_rows = pl.scan_parquet(FEATURES / "m1_features.parquet").select(pl.len()).collect().item()

figure_count = len(list(FIG.glob("*.png")))
table_count = len(list(TAB.glob("*.csv")))

print(f"raw CSV        : {len(csv_files)} files, {csv_bytes / 1e9:.2f} GB")
print(f"Parquet (zstd) : {len(parquet_files)} files, {parquet_bytes / 1e6:.0f} MB ({csv_bytes / parquet_bytes:.1f}x smaller)")
print(f"fact table     : {od_rows:,} rows, {od_trips:,} trips, {first_date} to {last_date}, {station_count} stations")
print(f"marts          : {len(mart_files)} tables, {mart_bytes / 1e6:.0f} MB, station_hour {station_hour_rows:,} rows")
print(f"M1 features    : {feature_rows:,} rows")
print(f"outputs        : {figure_count} figures, {table_count} tables")

raw CSV        : 13 files, 2.92 GB
Parquet (zstd) : 13 files, 115 MB (25.5x smaller)
fact table     : 112,520,054 rows, 1,194,454,788 trips, 2011-01-01 to 2023-05-02, 50 stations
marts          : 7 tables, 15 MB, station_hour 5,020,776 rows
M1 features    : 5,000,232 rows
outputs        : 73 figures, 8 tables


ข้อมูลเริ่มจาก CSV 13 ไฟล์ เก็บทุกการเดินทางระดับ ต้นทาง × ปลายทาง × ชั่วโมง ตลอด 12 ปีกว่า หลังตัดแถวซ้ำแล้วแปลงเป็น Parquet ไฟล์เล็กลงราว 25 เท่า ทุกการวิเคราะห์หลังจากนั้นอ่านจาก Parquet และ marts ไม่กลับไปอ่าน CSV อีก

ก่อนวิเคราะห์ ได้ตรวจคุณภาพข้อมูลและแก้ไว้ใน notebook 01-02 ทั้งหมด เช่น แถวซ้ำข้ามไฟล์ 16,078 แถว, วันที่ไม่มีข้อมูล 9 วัน (ใส่ค่าว่าง ไม่ใส่ 0), วัน strike 7 วัน และทริปหลงก่อนวันเปิดสถานี 462 แถว ผลรวมทริปของทุก mart ตรงกับ fact table

ช่วงเปรียบเทียบหลักของคำถามธุรกิจคือ ม.ค.-เม.ย. 2019 เทียบกับ ม.ค.-เม.ย. 2023 ใช้เดือนเดียวกันเพื่อตัดผลของฤดูกาล

## 2. คำตอบของคำถามธุรกิจ

### Q1 ผู้โดยสารฟื้นตัวแค่ไหน (notebook 04)
- วันทำงานอยู่ที่ 37% ของปี 2019 และแทบไม่ขยับตั้งแต่ ก.ย. 2022 (38.1% เป็น 38.9%) ส่วนวันหยุดฟื้นได้ 52.6%
- จันทร์ (34%) และศุกร์ (35%) ฟื้นน้อยกว่าอังคาร-พฤหัส (38-39%)
- ทริปไปและกลับ downtown SF เคยเป็น 67% ของทริปวันทำงาน ฟื้นแค่ 33% และคิดเป็น 72% ของทริปที่หายไป
- ทุก corridor ฟื้นได้ 32-43% ไม่มี corridor ไหนเกินครึ่ง
- ตรวจซ้ำด้วย counterfactual (notebook 13): SARIMA ที่เทรนเฉพาะข้อมูลก่อน COVID คาดว่าถ้าไม่มี COVID ต้นปี 2023 จะมี 389,620 คนต่อวันทำงาน ค่าจริง 151,848 คน คือฟื้น 39% ของระดับที่ควรเป็น (ช่วง 95%: 32-48%) ใกล้กับ 37% ที่เทียบกับปี 2019 เพราะก่อน COVID ผู้โดยสารลดลงอยู่แล้วตั้งแต่ปี 2016
- ทริปวันทำงานที่หายไปตั้งแต่ มี.ค. 2020 ถึง เม.ย. 2023 รวม 238 ล้านทริป นับเฉพาะวันทำงานปกติ

### Q2 รูปทรงของ peak เปลี่ยนไหม (notebook 05)
- ชั่วโมงที่ฟื้นน้อยที่สุดคือ 9:00 (30%) และ 18:00 (31%) ส่วน 11:00-14:00 ฟื้นได้ 46-50%
- peak แบนลง ชั่วโมงที่คนเยอะที่สุดเปลี่ยนจาก 8:00 เป็น 17:00 และ peak เย็นยังเร็วกว่าปี 2019 ~10 นาที (17:27 เทียบกับ 17:38)

### Q3 โหลดกระจุกที่ไหน (notebook 06)
- ชั่วโมงที่หนาแน่นที่สุดของ Transbay Tube ยังเป็น East Bay ไป SF ตอน 8:00 แต่เหลือ 7,948 ทริปต่อวันทำงาน (32% ของปี 2019)
- ช่วงเย็นยอดสูงสุดเลื่อนจาก 18:00 มา 17:00 และทิศสวนช่วงเช้ายังว่างกว่าทิศหลัก 6.9 เท่า
- คู่ OD ที่ฟื้นน้อยที่สุดคือการเดินทางไกลจากปลายสายเข้า downtown SF (DUBL และ PHIL กับ MONT/EMBR เหลือ 20-26%)

### Q4 สถานีใหม่สร้างผู้โดยสารใหม่จริงไหม (notebook 07)
- eBART (ANTC + PCTR): คนเข้าสถานีใหม่ 4,166 คนต่อวันทำงาน แต่เป็นผู้โดยสารใหม่สุทธิแค่ ~830-1,020 คน (20-24%) ที่เหลือย้ายมาจากสถานีเดิม
- Warm Springs: 3,075 คน เป็นผู้โดยสารใหม่สุทธิ ~860-1,070 คน (28-35%)
- สองกรณีที่เกิดก่อน COVID ให้ผลใกล้กัน สถานีปลายสายใหม่ได้ผู้โดยสารใหม่ราว 1 ใน 4 ถึง 1 ใน 3 ของยอดคนขึ้น

## 3. ผลของโมเดล

### M1 พยากรณ์ผู้โดยสารเข้าสถานีรายชั่วโมงล่วงหน้า 7 วัน (notebook 08)
ผลบน test (ม.ค. ถึง 2 พ.ค. 2023) ซึ่งใช้ครั้งเดียวตอนจบ เทียบกับวิธีเดิม 2 แบบ

In [3]:
m1_metrics = pd.read_csv(TAB / "m1_test_metrics.csv").set_index("model")
m1_metrics.round({"WAPE %": 2, "MAE": 2, "RMSE": 2, "R2": 3})

,WAPE %,MAE,RMSE,R2
model,,,,
baseline: last week,17.95,18.95,53.62,0.934
baseline: mean of 4 weeks,17.12,18.06,50.93,0.940
M1 point (Tweedie),14.50,15.30,44.41,0.955
M1 quantile P50,14.48,15.28,42.99,0.958


### M3 ทดสอบย้อนหลัง 8 ไตรมาส (notebook 10)
แต่ละไตรมาสเทรน M1 ใหม่ด้วยข้อมูลก่อนหน้าเท่านั้น แล้วพยากรณ์ทั้งไตรมาส เพื่อดูว่าผลบน test ไม่ได้บังเอิญ

In [4]:
backtest = pd.read_csv(TAB / "m3_backtest_by_quarter.csv").set_index("fold")
backtest

,M1 WAPE,baseline WAPE,error cut %,station-day M1,station-day baseline,P10-P90 coverage
fold,,,,,,
2021-Q3,14.31,15.91,10.1,6.80,9.59,80.3
2021-Q4,16.50,20.62,20.0,9.73,14.45,79.4
2022-Q1,15.09,17.83,15.4,10.23,13.51,80.0
2022-Q2,14.94,15.14,1.3,8.11,8.42,80.7
2022-Q3,13.41,16.12,16.8,6.60,9.50,82.2
2022-Q4,13.94,19.00,26.6,7.56,12.73,81.2
2023-Q1,15.32,18.79,18.5,9.91,13.74,78.2
2023-Q2,12.16,12.82,5.2,6.42,7.27,82.5
all 8 quarters,14.57,17.32,15.9,8.24,11.37,80.4


In [5]:
calibration = pd.read_csv(TAB / "m3_calibration.csv").set_index("segment")
calibration[["coverage before %", "coverage after %", "width change %"]]

,coverage before %,coverage after %,width change %
segment,,,
all hours,80.4,80.0,2.1
AM peak (workday 6-9),72.6,79.6,18.3
PM peak (workday 15-18),75.8,80.5,11.2
workday off-peak,82.8,79.9,-7.4
weekend / holiday,81.5,80.2,-3.4


- M1 ชนะค่าเฉลี่ย 4 สัปดาห์ทั้งบน test (WAPE 14.5% เทียบกับ 17.1%) และในการทดสอบย้อนหลังครบ 8 จาก 8 ไตรมาส (14.6% เทียบกับ 17.3%) ระดับสถานี-วัน 8.2% เทียบกับ 11.4%
- ชนะมากในไตรมาสที่มีวันหยุดเยอะ (2022-Q4 ลด error 27%) และชนะน้อยในไตรมาสที่ผู้โดยสารนิ่ง (2022-Q2 ลด 1%)
- ช่วง P10-P90 ของ M1 ช่วงเร่งด่วนแคบเกินไป หลังปรับด้วย error ของไตรมาสก่อน (M3) ครอบค่าจริงได้ ~80% ทุกช่วงเวลา โดยความกว้างรวมเพิ่มแค่ 2%

### M2 จัดกลุ่มสถานีตามบทบาท (notebook 09)
- สถานีแบ่งได้ 4 แบบที่เสถียร: ต้นทางคนทำงาน, ต้นทางแบบผสม, ย่านทำงาน และปลายทางทั้งวัน
- 22 จาก 30 สถานีต้นทางคนทำงานของปี 2019 กลายเป็นต้นทางแบบผสมในปี 2023 เหลือแบบเดิมแค่ 7 สถานีบนสาย Antioch และ Dublin
- อัตราส่วนวันหยุดต่อวันทำงานเพิ่มขึ้นทั้ง 48 สถานี (ค่ามัธยฐาน 0.33 เป็น 0.43)

### SARIMA counterfactual ของ Q1 (notebook 13)
โมเดล SARIMA(0,1,1)(1,1,0)12 บน log ผู้โดยสารเฉลี่ยต่อวันทำงานรายเดือน เทรนด้วยข้อมูล ม.ค. 2011 ถึง ก.พ. 2020 เท่านั้น แล้วพยากรณ์ว่าถ้าไม่มี COVID ช่วง มี.ค. 2020 ถึง เม.ย. 2023 จะเป็นเท่าไร
ตารางคือผลเมื่อเปลี่ยนโมเดล (การทดสอบความไว) โมเดลหลักคือแถว SARIMA(0, 1, 1)(1, 1, 0)12

In [6]:
sensitivity = pd.read_csv(TAB / "q1_counterfactual_sensitivity.csv").set_index("model")
sensitivity = sensitivity.drop(columns=["chosen"])
sensitivity

,no-COVID Jan-Apr 2023,recovery %,trips lost (million)
model,,,
"SARIMA(0, 1, 1)(1, 1, 1)12",394201,38.5,240.1
"SARIMA(0, 1, 1)(0, 1, 1)12",406200,37.4,245.3
"SARIMA(0, 1, 1)(0, 1, 2)12",389782,39.0,238.2
"SARIMA(0, 1, 1)(1, 1, 0)12",389620,39.0,238.4
"SARIMA(2, 1, 0)(0, 1, 2)12",383648,39.6,235.1
"SARIMA(1, 0, 0)(0, 1, 1)12 (no d)",403212,37.7,244.2
"SARIMA(0, 1, 1)(1, 1, 0)12 + outlier dummies",389291,39.0,238.6
"ETS (damped, mult.)",400410,37.9,242.4
flat at 2019 level (old method),409777,37.1,NaN


- เลือกโมเดลตามขั้นตอน Box-Jenkins: ADF และ KPSS ให้ d=1, D=1, ACF/PACF มี spike ที่ lag 1 และ 12 ค้นหา 117 โมเดล แล้วเลือกด้วยกฎที่เขียนไว้ก่อนดูผล
- พารามิเตอร์ทั้งสองตัวมีนัยสำคัญ (p < 0.01) รากอยู่นอกวงกลมหนึ่งหน่วย residual บน train ผ่าน 9 จาก 9 การทดสอบ
- 24 เดือนที่โมเดลไม่เคยเห็น (มี.ค. 2018 ถึง ก.พ. 2020): MAPE 3.3%, MASE 0.86 และช่วง 95% ครอบค่าจริงทั้ง 24 เดือน
- rolling CV 37 จุดเริ่ม: 1-4 เดือนข้างหน้าแม่นกว่า seasonal naive อย่างมีนัยสำคัญ แต่ที่ 24 เดือน error 7.6% และไม่ต่างจาก benchmark อย่างมีนัยสำคัญ (Diebold-Mariano) ผลจึงรายงานเป็นช่วง
- ทุกโมเดลที่ทดสอบให้การฟื้นตัว 37.1-39.6% ข้อสรุปจึงไม่ขึ้นกับการเลือกโมเดล

## 4. ข้อเสนอสำหรับผู้บริหาร

เรียงจากเรื่องที่กระทบต้นทุนมากที่สุด แต่ละข้อมีที่มาในวงเล็บ

1. วางงบและบริการโดยถือว่าวันทำงาน ~40% ของปี 2019 คือระดับปกติใหม่ระยะสั้น ตัวเลขนิ่งมาตั้งแต่ ก.ย. 2022 จึงไม่ควรรอให้ผู้โดยสารกลับมาเอง และไม่ควรตั้งเป้า "กลับไปเท่าปี 2019" เพราะก่อน COVID ผู้โดยสารก็ลดลงอยู่แล้ว เทียบกับระดับที่ควรเป็นถ้าไม่มี COVID ได้ 39% (Q1, notebook 13)

2. ลดขนาดขบวนช่วง 8:00-9:59 และ 18:00-19:59 ก่อนช่วงอื่น แต่ไม่ลดกลางวันและวันหยุดในสัดส่วนเดียวกัน สองช่วงนี้ฟื้นแค่ 30-31% ขณะที่กลางวันฟื้น 46-50% และวันหยุดสำคัญขึ้นทุกสถานี (Q1, Q2, M2)

3. ย้ายขบวนเสริมช่วงเย็นของ Transbay Tube จาก 18:00 มา 17:00 และใช้ขบวนสั้นในทิศสวน ช่วงเช้าทิศสวนว่างกว่าทิศหลัก 6.9 เท่า (Q3)

4. สาย Antioch และ Dublin ให้ลดจำนวนตู้ แต่คงเวลาออกขบวนช่วงเช้าไว้ สองผลนี้ต้องอ่านคู่กัน คือคู่ OD จากสองสายนี้เข้า downtown เหลือแค่ 20-26% (Q3) แต่สถานีบนสองสายนี้ยังเป็นต้นทางคนทำงานที่ peak เช้าแหลมอยู่ (M2) แปลว่าคนน้อยลงแต่ยังมาเวลาเดิม

5. ใช้ M1 แทนค่าเฉลี่ยย้อนหลังในการวางแผนรายสัปดาห์ ใช้ P50 เป็นแผนหลัก และใช้ P90 ที่ปรับแล้วจาก M3 กำหนดขบวนสำรอง เทรนใหม่ทุกไตรมาสด้วยค่าเดิมได้ ใช้เวลา ~4 นาทีบน GPU (M1, M3)

6. ทำปฏิทินงานใหญ่ของเมืองและตารางงานที่ Coliseum วันที่พลาดมากที่สุดคืองานระดับเมือง (พาเหรด +48%, Pride +80%) และ COLS มีวันคนล้นมากที่สุด 103 วัน ส่วนใหญ่ช่วงฤดูกาลกีฬาและหลัง 17:00 สัปดาห์ที่มีงานควรให้คนปรับแผนก่อนใช้ค่าพยากรณ์ (M3)

7. ประเมินโครงการขยายสายด้วยผู้โดยสารใหม่สุทธิ ไม่ใช่ยอดคนขึ้นที่สถานีใหม่ ผู้โดยสารใหม่จริงเป็นแค่ 1 ใน 4 ถึง 1 ใน 3 ของยอดคนขึ้น ที่เหลือย้ายมาจากสถานีเดิม (Q4)

8. ให้ฝ่ายปฏิบัติการตรวจวันที่สถานีมีคนเข้าเกือบเป็นศูนย์ เช่น MLBR 12-13 และ 22-29 ธ.ค. 2021 ว่าปิดสถานีหรือข้อมูลประตูกั้นขาด (M3)

## 5. ตาราง Insight → Pain point รวม

รวมจากตารางท้าย notebook 04-10 และ 13 จัดกลุ่มตามการตัดสินใจของ BART 3 เรื่อง คือ บริการ ตารางเดินรถ และทรัพยากร และอีก 2 เรื่องคือ การลงทุนขยายสาย และเครื่องมือวางแผน

### บริการ: คงหรือลดบริการตรงไหน

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ | ที่มา |
|---|---|---|---|---|---|
| 1 | วันทำงานอยู่ที่ 37% ของปี 2019 (39% ของระดับที่ควรเป็นถ้าไม่มี COVID ช่วง 32-48%) และนิ่งตั้งแต่ ก.ย. 2022 ทริปวันทำงานหายไปสะสม 238 ล้านทริป | รายได้ที่หายไป | ตรงจุดเรื่องระดับ แต่แปลงเป็นรายได้ไม่ได้ เพราะไม่มีข้อมูลค่าโดยสาร | ถือว่า ~40% คือระดับปกติใหม่ระยะสั้น | Q1, 13 |
| 2 | วันหยุดฟื้น 52.6% และอัตราส่วนวันหยุดต่อวันทำงานเพิ่มขึ้นทั้ง 48 สถานี | คงหรือลดบริการวันหยุด | ตรงจุด | ไม่ลดบริการวันหยุดตามสัดส่วนของวันทำงาน | Q1, M2 |
| 3 | จันทร์และศุกร์ฟื้น 34-35% อังคาร-พฤหัส 38-39% วันจันทร์คนน้อยกว่าวันพุธ ~23,500 คน | ขนาดบริการรายวัน | ตรงจุด | ขบวนสั้นลงวันจันทร์และศุกร์ | Q1 |
| 4 | ทริป downtown SF คิดเป็น 72% ของทริปที่หายไป ฟื้นแค่ 33% และสถานีย่านทำงานของปี 2019 ฟื้น 31.8% ต่ำกว่าระบบ (36.6%) | ช่องว่างอยู่ที่ไหน | ตรงจุด | การตลาดและค่าโดยสารสำหรับคนทำงาน hybrid (ยังพิสูจน์ผลด้วยข้อมูลชุดนี้ไม่ได้) | Q1, M2 |
| 5 | ทุก corridor ฟื้น 32-43% สนามบินและปลายสายฟื้นดีที่สุด (SFIA 60%) | ตัดบริการราย corridor | บางส่วน | ปรับตามช่วงเวลาและวันแทนการตัดทั้ง corridor | Q1 |

### ตารางเดินรถ: ช่วงเวลาไหนต้องปรับ

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ | ที่มา |
|---|---|---|---|---|---|
| 6 | 9:00 และ 18:00 ฟื้นแค่ 30-31% กลางวันฟื้น 46-50% | รถเกินช่วง peak | ตรงจุด แต่บอกจำนวนขบวนที่เกินไม่ได้ เพราะไม่มีข้อมูลความจุ | ลดขนาดขบวน 8:00-9:59 และ 18:00-19:59 ก่อน | Q2 |
| 7 | peak แบนลง ชั่วโมงที่คนเยอะที่สุดเปลี่ยนจาก 8:00 เป็น 17:00 สัดส่วนคนเข้าช่วงเช้าลดลงที่ 37 จาก 38 สถานีต้นทาง | ตารางที่ออกแบบให้ peak เช้าแหลม | ตรงจุด | กระจายความถี่ให้สม่ำเสมอขึ้นตลอดวัน | Q2, M2 |
| 8 | peak เย็นยังเร็วกว่าปี 2019 ~10 นาที และค่อยๆ เลื่อนกลับ | เวลาเริ่มบริการช่วง peak | ตรงจุด | เริ่มเพิ่มรถช่วงเย็นเร็วขึ้นเล็กน้อย และติดตามต่อ | Q2 |
| 9 | COLS ช่วง 22:00-23:59 เหลือ 9-14% ของปี 2019 | บริการรอบดึก | บางส่วน สาเหตุ (ทีมกีฬาย้าย) มาจากความรู้ภายนอก | ทบทวนบริการเสริมหลังงานที่ Coliseum | Q2 |

### ทรัพยากร: ขบวน ตู้ และพนักงาน

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ | ที่มา |
|---|---|---|---|---|---|
| 10 | ชั่วโมง-ทิศที่หนาแน่นที่สุดของ Tube ยังเป็น East Bay ไป SF 8:00 เหลือ 7,948 ทริป (32% ของปี 2019) | ชั่วโมงที่กำหนดความจุ | ตรงจุด | ใช้ 8:00 ทิศตะวันตกกำหนดขนาดขบวน ความต้องการสูงสุดเหลือราว 1 ใน 3 | Q3 |
| 11 | ยอดเย็นสูงสุดย้ายจาก 18:00 มา 17:00 ส่วน 18:00 เหลือ 27% | ขบวนเสริมช่วงเย็น | ตรงจุด | ย้ายขบวนเสริมมา 17:00 | Q3 |
| 12 | ทิศสวนช่วงเช้าว่างกว่าทิศหลัก 6.9 เท่า (ช่วงเย็น 3.7 เท่า) | ขบวนเปล่าตอนกลับทิศ | ตรงจุด | ขบวนทิศสวนใช้ขบวนสั้นลง | Q3 |
| 13 | คู่ OD ไกลจาก DUBL และ PHIL เข้า MONT/EMBR เหลือ 20-26% แต่ 7 สถานีบนสาย Antioch และ Dublin ยังเป็นต้นทางคนทำงาน คนเข้าช่วงเช้า 52-70% | ลดขบวนยาวที่ไหน | ตรงจุดเมื่ออ่านสองผลคู่กัน | ลดจำนวนตู้ แต่คงเวลาออกขบวนช่วงเช้า | Q3, M2 |
| 14 | สถานีแบ่งได้ 4 แบบที่เสถียร และ 22 จาก 30 ต้นทางคนทำงานเปลี่ยนเป็นต้นทางแบบผสม | แผนพนักงานแบบเดียวทุกสถานี | ตรงจุด | วางแม่แบบพนักงานแยกตาม 4 แบบ | M2 |
| 15 | WARM เปลี่ยนจากต้นทางเป็นปลายทางตอนเช้า เพราะผู้โดยสารต้นทางย้ายไป MLPT/BERY | พนักงานรายสถานี | บางส่วน เป็นรายสถานี | ทบทวนพนักงานช่วงเช้าของ WARM ในฐานะสถานีปลายทาง | M2, Q4 |

### การลงทุนขยายสาย

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ | ที่มา |
|---|---|---|---|---|---|
| 16 | eBART ได้ผู้โดยสารใหม่สุทธิ 20-24% ของยอดคนขึ้น PITT เสียไป 39% | ได้ผู้โดยสารใหม่จริงเท่าไร | ตรงจุด เป็นกรณีก่อน COVID | ใช้ผู้โดยสารใหม่สุทธิประเมินโครงการ | Q4 |
| 17 | Warm Springs ได้ผู้โดยสารใหม่สุทธิ 28-35% FRMT เสียไป 25% | ได้ผู้โดยสารใหม่จริงเท่าไร | ตรงจุด | ใช้เป็นตัวเลขอ้างอิงของสถานีปลายสายใหม่ | Q4 |
| 18 | Silicon Valley เปิดช่วง COVID วัดผู้โดยสารใหม่สุทธิไม่ได้ แต่รูปแบบสอดคล้องกับผู้โดยสาร WARM ย้ายไป | ประเมินโครงการล่าสุด | บางส่วน | ดูทั้งกลุ่มสถานี อย่าตัดบริการ WARM จากตัวเลขการฟื้นตัวอย่างเดียว | Q4 |

### เครื่องมือวางแผนรายสัปดาห์และวันพิเศษ

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ | ที่มา |
|---|---|---|---|---|---|
| 19 | M1 WAPE 14.5% เทียบกับ 17.1% บน test และชนะ 8 จาก 8 ไตรมาสในการทดสอบย้อนหลัง | ความแม่นของแผนรายสัปดาห์ | ตรงจุด | ใช้ M1 P50 เป็นแผนหลัก | M1, M3 |
| 20 | ระดับสถานี-วัน WAPE 8.2% เทียบกับ 11.4% (ย้อนหลัง 8 ไตรมาส) | วางพนักงานรายวัน | ตรงจุด | ใช้ผลรวมรายวันของ M1 วางพนักงานสถานี | M1, M3 |
| 21 | หลังปรับ ช่วง P10-P90 ช่วงเช้าครอบค่าจริงจาก 72.6% เป็น 79.6% ทั้งวัน 80.0% ความกว้างรวมเพิ่มแค่ 2% | ขบวนสำรองช่วงเร่งด่วน | ตรงจุด แต่รายไตรมาสยังแกว่ง 73-87% | ใช้ P90 ที่ปรับแล้วกำหนดขบวนสำรอง | M3 |
| 22 | งานระดับเมืองทำให้พลาดมากที่สุด (พาเหรด Warriors +48% ใน 35 สถานี, Pride +80%, พายุ -42%) | เตรียมรถวันพิเศษ | บางส่วน สาเหตุมาจากความรู้ภายนอก และโมเดลไม่รู้ล่วงหน้า | ทำปฏิทินงานใหญ่ของเมือง | M3 |
| 23 | COLS มีวันคนล้น 103 วัน 86% อยู่ช่วง เม.ย.-ต.ค. และ 61% ของคนที่เกินเข้าหลัง 17:00 | ขบวนหลังงานกีฬา | ตรงจุด | นำตารางงานที่ Coliseum เข้าโมเดล | M3 |
| 24 | หลังสัปดาห์ที่มีวันหยุดและพายุ (ม.ค. 2023) ค่าจริงสูงกว่าพยากรณ์ 31-42% เพราะค่าอ้างอิง 4 สัปดาห์ต่ำผิดปกติ | ความแม่นหลังช่วงผิดปกติ | ตรงจุด เป็นจุดอ่อนของ M1 | ให้ lag ข้ามวันหยุดและวันพายุ | M3 |
| 25 | MLBR คนเข้าเกือบเป็นศูนย์ 14 วัน เช่น 12-13 และ 22-29 ธ.ค. 2021 | ข้อมูลและการปิดสถานี | บางส่วน บอกสาเหตุไม่ได้ | ให้ฝ่ายปฏิบัติการตรวจ | M3 |
| 26 | SARIMA รายเดือนพยากรณ์ 1-4 เดือนข้างหน้าคลาด 1.2-2.3% แต่ 2 ปีข้างหน้าคลาดเฉลี่ย 7.6% | วางงบระยะยาว | ตรงจุดเรื่องข้อจำกัด | ใช้พยากรณ์รายไตรมาส ส่วนแผนหลายปีใช้ scenario และช่วง ไม่ใช่ตัวเลขเดียว | 13 |

## 6. ทำงานได้เร็วแค่ไหน

ทุกขั้นที่หนักครอบด้วย `stage()` จาก `src/bart/perf.py` ซึ่งบันทึกเวลาจริง (wall), เวลา CPU, จำนวน core ที่ใช้จริง (CPU time / wall time), RAM สูงสุดของ process และหน่วยความจำ GPU ลง `outputs/logs/perf.jsonl`

วิธีอ่านไฟล์ log
- ไฟล์เก็บทุกครั้งที่รัน จึงใช้ค่าล่าสุดของแต่ละขั้น
- ใช้เฉพาะชื่อขั้นที่ยังอยู่ใน notebook ปัจจุบัน (ขั้นที่เคยทดสอบแล้วเปลี่ยนชื่อไม่นับ)
- `10.backtest` ถ้ารันซ้ำจะโหลดผลที่เก็บไว้และจบในไม่กี่วินาที จึงใช้ครั้งล่าสุดที่เทรนจริง (นานกว่า 60 วินาที)
- หน่วยความจำ GPU วัดด้วย `nvidia-smi` เป็นค่าของทั้งการ์ด รวมโปรแกรมอื่นที่เปิดอยู่ด้วย

In [7]:
CURRENT_STAGES = [
    "00.gpu_test", "00.raw_csv_profile",
    "01.csv_to_parquet", "01.validate",
    "02.calendar", "02.station_hour", "02.station_day_system_hour", "02.od_month", "02.write_marts",
    "03.overview", "03.sparsity", "03.trip_buckets", "03.monthly_trend", "03.surge_days", "03.station_period",
    "03.od_period", "03.transbay", "03.lag_correlation", "03.baselines",
    "04.recovery_index", "04.station_recovery",
    "05.hourly", "05.peak_centre", "05.station_hour_recovery",
    "06.crossing_hour_from_fact",
    "07.ebart", "07.warm_springs", "07.silicon_valley",
    "08.build_features", "08.write_features", "08.prepare_inputs",
    "08.variant_all", "08.variant_weighted", "08.variant_post_covid",
    "08.optuna_10_trials", "08.optuna_full", "08.learning_rate_check", "08.final_models", "08.shap",
    "09.period_hours", "09.choose_k", "09.final_kmeans",
    "10.backtest",
]
CACHED_STAGES = ["10.backtest"]
CACHE_MIN_SECONDS = 60
SKLEARN_STAGES = ["09.choose_k", "09.final_kmeans"]

NOTEBOOK_NAMES = {
    "00": "00 setup", "01": "01 ingest", "02": "02 marts", "03": "03 EDA",
    "04": "04 Q1 recovery", "05": "05 Q2 peak shape", "06": "06 Q3 load flows", "07": "07 Q4 new stations",
    "08": "08 M1 forecast", "09": "09 M2 typology", "10": "10 M3 backtest",
}

latest = {}
with open(LOGS / "perf.jsonl") as handle:
    for line in handle:
        record = json.loads(line)
        name = record["stage"]
        if name not in CURRENT_STAGES:
            continue
        if name in CACHED_STAGES and record["wall_s"] < CACHE_MIN_SECONDS:
            continue
        # the log is appended in time order, so a later run replaces an earlier one
        latest[name] = record


def engine_of(record):
    if "peak_gpu_mem_mb" in record:
        return "XGBoost on GPU"
    if record["stage"] in SKLEARN_STAGES:
        return "scikit-learn on CPU"
    return "DuckDB / Polars on CPU"


stage_rows = []
for name in CURRENT_STAGES:
    if name not in latest:
        print(f"no record yet for {name}")
        continue
    record = latest[name]
    gpu_gb = None
    if "peak_gpu_mem_mb" in record:
        gpu_gb = record["peak_gpu_mem_mb"] / 1000
    notebook_key = name.split(".")[0]
    stage_rows.append({
        "notebook": NOTEBOOK_NAMES[notebook_key],
        "stage": name,
        "engine": engine_of(record),
        "seconds": record["wall_s"],
        "cores": record["effective_cores"],
        "peak_ram_gb": record["peak_rss_gb"],
        "gpu_mem_gb": gpu_gb,
        "run_at": record["ts"],
    })

stages = pl.DataFrame(stage_rows)
print(f"stages with a record: {stages.height} of {len(CURRENT_STAGES)}")

stages with a record: 43 of 43


### เวลาและทรัพยากรราย notebook

นับเฉพาะขั้นที่ครอบด้วย `stage()` ไม่รวมเวลาวาดกราฟและเซฟรูป

In [8]:
per_notebook = (
    stages.group_by("notebook")
    .agg(
        pl.len().alias("stages"),
        pl.col("seconds").sum().alias("seconds"),
        pl.col("cores").max().alias("max cores"),
        pl.col("peak_ram_gb").max().alias("peak RAM GB"),
        pl.col("gpu_mem_gb").max().alias("peak GPU mem GB"),
    )
    .sort("notebook")
)

total_seconds = float(stages["seconds"].sum())
gpu_seconds = float(stages.filter(pl.col("engine") == "XGBoost on GPU")["seconds"].sum())
cpu_seconds = total_seconds - gpu_seconds

print(f"all stages      : {total_seconds / 60:.1f} min")
print(f"GPU model stages: {gpu_seconds / 60:.1f} min ({100 * gpu_seconds / total_seconds:.1f}%)")
print(f"everything else : {cpu_seconds:.1f} s")

per_notebook_table = per_notebook.to_pandas().set_index("notebook")
per_notebook_table["minutes"] = per_notebook_table["seconds"] / 60
per_notebook_table = per_notebook_table[["stages", "seconds", "minutes", "max cores", "peak RAM GB", "peak GPU mem GB"]]
per_notebook_table.round(2)

all stages      : 90.4 min
GPU model stages: 89.9 min (99.4%)
everything else : 32.5 s


,stages,seconds,minutes,max cores,peak RAM GB,peak GPU mem GB
notebook,,,,,,
00 setup,2,4.34,0.07,17.63,1.92,1.12
01 ingest,2,14.73,0.25,16.46,8.57,NaN
02 marts,5,3.93,0.07,18.74,1.57,NaN
03 EDA,10,2.90,0.05,17.80,9.83,NaN
04 Q1 recovery,2,0.01,0.00,6.37,0.24,NaN
05 Q2 peak shape,3,0.03,0.00,6.99,0.26,NaN
06 Q3 load flows,1,0.21,0.00,10.36,0.28,NaN
07 Q4 new stations,3,0.02,0.00,3.49,0.26,NaN
08 M1 forecast,11,3477.69,57.96,18.63,10.24,3.76


In [9]:
bar_rows = per_notebook.sort("notebook", descending=True)

def time_label(seconds):
    if seconds >= 60:
        return f"{seconds / 60:.1f} min"
    if seconds >= 0.1:
        return f"{seconds:.1f} s"
    return f"{seconds * 1000:.0f} ms"


labels = []
for seconds in bar_rows["seconds"].to_list():
    labels.append(time_label(seconds))

colors = []
for notebook in bar_rows["notebook"].to_list():
    if notebook.startswith("08") or notebook.startswith("10"):
        colors.append(viz.CONTRAST)
    else:
        colors.append(viz.ACCENT)

fig = go.Figure(go.Bar(
    x=bar_rows["seconds"], y=bar_rows["notebook"], orientation="h",
    marker=dict(color=colors), text=labels, textposition="outside", cliponaxis=False,
    hovertemplate="%{y}: %{text}<extra></extra>",
))
fig.update_xaxes(type="log", title_text="seconds of measured work (log scale)", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False)
fig.update_layout(hovermode="closest", showlegend=False, margin=dict(l=170, r=90))

title = f"Data work takes seconds; GPU model training is {100 * gpu_seconds / total_seconds:.0f}% of the total time"
subtitle = f"Measured time per notebook, {total_seconds / 60:.0f} min in all; orange = notebooks that train XGBoost on the GPU"
viz.titled(fig, title, subtitle)
viz.save(fig, "11_time_per_notebook", height=560)
fig

### งานแต่ละแบบใช้เครื่องต่างกัน

แต่ละจุดคือหนึ่งขั้น แกนนอนคือเวลา แกนตั้งคือจำนวน core ของ CPU ที่ใช้จริง

In [10]:
ENGINE_COLORS = {
    "DuckDB / Polars on CPU": viz.ACCENT,
    "XGBoost on GPU": viz.CONTRAST,
    "scikit-learn on CPU": viz.SERIES[2],
}

fig = go.Figure()
for engine, color in ENGINE_COLORS.items():
    part = stages.filter(pl.col("engine") == engine)
    if part.height == 0:
        continue
    fig.add_trace(go.Scatter(
        x=part["seconds"], y=part["cores"], mode="markers", name=engine,
        marker=dict(color=color, size=11, line=dict(color="white", width=2)),
        text=part["stage"],
        hovertemplate="%{text}<br>%{x:.2f} s, %{y:.1f} cores<extra></extra>",
    ))

LABELLED = {
    "01.csv_to_parquet": (40, -30),
    "02.station_hour": (-40, -35),
    "08.build_features": (60, 30),
    "08.optuna_full": (-170, -35),
    "10.backtest": (-70, -95),
    "09.choose_k": (40, -25),
}
for name, offset in LABELLED.items():
    row = stages.filter(pl.col("stage") == name)
    if row.height == 0:
        continue
    seconds = row["seconds"][0]
    # on a log axis plotly places annotations by log10 of the value
    fig.add_annotation(
        x=math.log10(seconds), y=float(row["cores"][0]), xref="x", yref="y",
        text=f"{name} ({time_label(seconds)})", showarrow=True, arrowhead=0, arrowcolor=viz.MUTED,
        ax=offset[0], ay=offset[1], font=dict(size=11, color=viz.INK2),
    )

# plotly needs log10 of the range on a log axis
fig.update_xaxes(type="log", title_text="wall time in seconds (log scale)", range=[-2.7, 3.6])
fig.update_yaxes(title_text="CPU cores in use", range=[0, 20])
fig.update_layout(hovermode="closest")
title = "Data steps use most of the 20 CPU threads; model training runs on the GPU"
heavy_data = stages.filter((pl.col("engine") == "DuckDB / Polars on CPU") & (pl.col("seconds") >= 0.4))
gpu_stages = stages.filter(pl.col("engine") == "XGBoost on GPU")
subtitle = (
    f"Each dot is one measured stage; data steps longer than 0.4 s use "
    f"{heavy_data['cores'].min():.0f}-{heavy_data['cores'].max():.0f} cores, "
    f"GPU stages keep {gpu_stages['cores'].min():.0f}-{gpu_stages['cores'].max():.0f} busy"
)
viz.titled(fig, title, subtitle)
viz.save(fig, "11_stage_engines", height=560)
fig

### ตัวอย่างความเร็วของแต่ละงาน

คำนวณจากเวลาใน log กับขนาดข้อมูลจริง

In [11]:
def stage_seconds(name):
    return stages.filter(pl.col("stage") == name)["seconds"][0]


with open(MODELS / "m1_best_params.json") as handle:
    m1 = json.load(handle)

ingest_seconds = stage_seconds("01.csv_to_parquet")
marts_seconds = float(stages.filter(pl.col("notebook") == "02 marts")["seconds"].sum())
features_seconds = stage_seconds("08.build_features") + stage_seconds("08.write_features")
backtest_seconds = stage_seconds("10.backtest")

FOLDS = 8
QUANTILES = 3
trees_per_fold = m1["point_rounds"] + m1["quantile_rounds"] * QUANTILES
trees_total = trees_per_fold * FOLDS

print(f"CSV to Parquet   : {od_rows / 1e6:.1f} M rows in {ingest_seconds:.1f} s = {od_rows / ingest_seconds / 1e6:.1f} M rows per second")
print(f"                   {csv_bytes / 1e9:.2f} GB of CSV in {ingest_seconds:.1f} s = {csv_bytes / ingest_seconds / 1e6:.0f} MB per second")
print(f"all 7 marts      : {marts_seconds:.1f} s, including {station_hour_rows:,} zero-filled station-hours")
print(f"M1 features      : {feature_rows:,} rows in {features_seconds:.1f} s (Polars build + write)")
print(f"M3 backtest      : {FOLDS} quarters x {trees_per_fold:,} trees = {trees_total:,} trees in {backtest_seconds / 60:.1f} min")
print(f"                   {trees_total / backtest_seconds:.1f} trees per second on 4.1-4.9 M training rows each")

CSV to Parquet   : 112.5 M rows in 9.2 s = 12.3 M rows per second
                   2.92 GB of CSV in 9.2 s = 319 MB per second
all 7 marts      : 3.9 s, including 5,020,776 zero-filled station-hours
M1 features      : 5,000,232 rows in 2.8 s (Polars build + write)
M3 backtest      : 8 quarters x 5,697 trees = 45,576 trees in 31.9 min
                   23.8 trees per second on 4.1-4.9 M training rows each


สรุปความเร็ว (จาก log ของรันวันที่ 12-13 ต.ค. 2026)
- ทั้งงานวัดได้ ~75 นาที บนโน้ตบุ๊กเครื่องเดียว ไม่ต้องใช้ cluster
- ทุกขั้นที่ไม่ได้เทรน XGBoost รวมกันใช้แค่ ~32 วินาที เช่น แปลง CSV 2.92 GB (112.5 ล้านแถว) เป็น Parquet ใน 7.9 วินาที ตรวจทุกแถวอีก 5.5 วินาที สร้าง marts ครบ 7 ตาราง 4.2 วินาที และสร้าง feature ของ M1 5 ล้านแถว 3.1 วินาที
- ขั้นข้อมูลที่หนักใช้ CPU จริง 14-18 core จาก 20 thread แปลว่า DuckDB และ Polars กระจายงานได้เกือบเต็มเครื่อง
- เวลา 99% อยู่ที่การเทรน XGBoost บน GPU: จูน 50 trial ~28 นาที และทดสอบย้อนหลัง 8 ไตรมาส ~32 นาที (สร้างต้นไม้ 45,576 ต้น ~24 ต้นต่อวินาที บนข้อมูลเทรน 4-5 ล้านแถวต่อไตรมาส)
- RAM ของ process สูงสุด ~11 GB จาก 32 GB (เทรนโมเดลสุดท้ายของ M1) หน่วยความจำ GPU สูงสุด ~4.5 GB จาก 8 GB จึงไม่มีขั้นไหนใช้หน่วยความจำเกินเครื่อง

ระหว่างทดสอบย้อนหลัง (M3) ได้ดูเครื่องด้วย `nvidia-smi` และ `sensors` เพิ่ม ตัวเลขชุดนี้ไม่ได้อยู่ใน `perf.jsonl`
- GPU ใช้งาน 98% กินไฟ ~70 W ความเร็ว ~2,490 MHz ใช้ VRAM ~4 GB
- CPU ทั้งเครื่องว่าง ~92% มีแค่ 1 core ทำงานเต็ม คือ thread หลักของ XGBoost ที่สั่งงาน GPU จึงเห็นขั้น GPU ใช้ CPU แค่ ~1 core ในกราฟ
- อุณหภูมิ core ที่ทำงาน ~90°C (ขีดจำกัด 100°C) GPU 63°C พอเปิดพัดลมรองโน้ตบุ๊ก GPU ลดเหลือ 60°C CPU 86-89°C และไม่มีการลดความเร็วจากความร้อน
- เครื่องเสียบสายชาร์จและตั้ง power profile เป็น performance ตลอดงาน

ไม่ได้ทดลองเทรนแบบเดียวกันบน CPU เพื่อเทียบ (ตั้งใจไม่ทำ benchmark เทียบเครื่องมือ) จึงไม่ได้บอกว่า GPU เร็วกว่ากี่เท่า

## 7. เทคนิค Big Data ที่ใช้

แต่ละข้อบอกว่าใช้ที่ไหน ทำไม และได้ผลอะไร

### การเก็บข้อมูล
- Parquet แบบ zstd เรียงตาม date, hour, origin, dest และเขียนทีละปี ไฟล์เล็กลง 25.5 เท่า row group ละประมาณ 1 เดือน ไม่มีช่วงวันที่ทับกันเลย (118 row group) query ที่กรองวันที่จึงข้าม row group ที่ไม่เกี่ยวได้ (notebook 01)
- กำหนด type เอง สถานีเป็น ENUM 50 ค่า trips เป็น USMALLINT ไม่ให้ DuckDB เดา type (notebook 01)

### ความถูกต้องของข้อมูล
- ตรวจแถวซ้ำข้ามไฟล์ เทียบจำนวนแถวและทริปกับ CSV ทุกปี และตรวจผลรวมทริปของทุก mart ให้ตรงกับ fact table (notebook 01-02)
- marts เติมศูนย์ครบทุกสถานี-ชั่วโมง แต่วันที่ไม่มีข้อมูลใส่ค่าว่าง เพื่อไม่ให้ค่าเฉลี่ยและ lag ของโมเดลเพี้ยน (notebook 02)

### การประมวลผล
- DuckDB ทำ SQL ขนานทุก thread จำกัดหน่วยความจำ 20 GB และให้ spill ลงดิสก์ได้ ขั้นที่หนักใช้ CPU จริง 14-18 core
- Polars ทำ feature ของ M1 5 ล้านแถว ในไม่กี่วินาที
- เลือกเครื่องตามงาน: XGBoost บน GPU สำหรับข้อมูลหลายล้านแถวที่ต้องเทรนหลายรอบ ส่วน clustering 96 ตัวอย่างใช้ CPU เพราะ GPU จะมีแต่ overhead
- `QuantileDMatrix` (256 bins) ทำให้ข้อมูลเทรน 4-5 ล้านแถวพอดีกับ VRAM 8 GB

### ความถูกต้องของโมเดล
- ทุก lag ย้อนหลังอย่างน้อย 7 วัน และสุ่มแถวมาตรวจว่า lag ตรงกับค่าจริง เพื่อกัน data leakage (notebook 08)
- แบ่ง train / valid / test ตามเวลา และใช้ test ครั้งเดียวตอนจบ (notebook 08)
- ทดสอบย้อนหลังแบบ rolling-origin 8 ไตรมาส และปรับช่วง P10-P90 ด้วย error ของไตรมาสก่อนหน้าเท่านั้น (notebook 10)
- ตรวจความเสถียรของผล: ARI ข้าม 50 seed และเทียบกับ Ward ใน M2, ลองเกณฑ์ตัดสินเหตุการณ์ 15 แบบใน M3
- SARIMA ตามขั้นตอน Box-Jenkins ครบ: ทดสอบ stationarity สองทาง, อ่าน ACF/PACF, ค้นหา 117 โมเดลด้วย AICc, ตรวจพารามิเตอร์ (z, LR test, Wald) และ residual 9 การทดสอบ, rolling-origin CV กับ benchmark พร้อม Diebold-Mariano และทดสอบความไวข้าม 9 โมเดล (notebook 13)

## 8. ข้อจำกัดและงานต่อ

ข้อจำกัดของทั้งงาน
- ไม่มีข้อมูลค่าโดยสาร ทุกตัวเลขเป็นจำนวนทริป แปลงเป็นรายได้ไม่ได้
- ไม่มีตารางเดินรถและความจุขบวน ตัวเลขทั้งหมดคือความต้องการ ไม่ใช่ความแออัด
- feed มีชั่วโมงเดียวต่อทริป (ชั่วโมงที่เข้าสถานี) โหลดใน Tube จริงอาจเหลื่อมไปชั่วโมงถัดไป
- ปี 2023 มีข้อมูลแค่ ม.ค. ถึง 2 พ.ค. ภาพการฟื้นตัวจึงเป็นระยะสั้น
- สาเหตุของเหตุการณ์ระดับเมือง (พาเหรด, Pride, พายุ) มาจากความรู้ภายนอก ไม่ได้ยืนยันจากข้อมูลชุดนี้
- โมเดลพยากรณ์ entries อย่างเดียว ไม่ได้พยากรณ์ exits หรือโหลดบนขบวน
- counterfactual ของ Q1 ต้องพยากรณ์ไกล 38 เดือนจากข้อมูลก่อน COVID ซึ่งเป็นระยะที่ไม่มีโมเดลไหนแม่นกว่าวิธีง่ายอย่างมีนัยสำคัญ จึงรายงานเป็นช่วง และทำได้เฉพาะยอดรวมทั้งระบบ ตัวเลขรายสถานีและรายชั่วโมงยังเทียบกับปี 2019

งานต่อที่คุ้มที่สุด
1. เพิ่มปฏิทินงานใหญ่ของเมืองและตารางงานที่ Coliseum เป็น feature ของ M1
2. ให้ lag ของ M1 ข้ามวันหยุดและวันพายุ แก้จุดอ่อนช่วง ม.ค. 2023
3. เพิ่มข้อมูลค่าโดยสารเพื่อแปลงผล Q1 เป็นรายได้ และข้อมูลตารางเดินรถเพื่อคำนวณความแออัดบนขบวน